# Retrain `gradient_boosting` on 400-feature pipeline

**Steps:**
1. Upload `train.csv` from your `dataset/` folder when prompted
2. Upload `scaler.pkl` from your project root when prompted
3. Run all cells — takes ~5 min on Colab GPU
4. Download `gradient_boosting_model.pkl` at the end
5. Drop it into your local `models/` folder and restart `app.py`

In [ ]:
# ── Install dependencies ─────────────────────────────────────────────────────
!pip install sentence-transformers scikit-learn --quiet

In [ ]:
# ── Upload files ─────────────────────────────────────────────────────────────
from google.colab import files

print('Upload train.csv (from your dataset/ folder)')
uploaded = files.upload()   # select train.csv

print('\nUpload scaler.pkl (from your project root)')
uploaded2 = files.upload()  # select scaler.pkl

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import pickle, re, time, warnings
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sentence_transformers import SentenceTransformer

warnings.filterwarnings('ignore')
RANDOM_SEED = 42
print('Imports OK')

In [ ]:
# ── Feature extraction (must match app.py exactly) ───────────────────────────
def _extract_float(text, prefix):
    idx = text.find(prefix)
    if idx == -1: return 0.0
    snippet = text[idx+len(prefix):idx+len(prefix)+30]
    m = re.search(r'[\d]+\.?[\d]*', snippet)
    return float(m.group()) if m else 0.0

def _extract_pack(text):
    for pat in [r'pack\s+of\s+(\d+)', r'\((\d+)\s+pack\)',
                r'(\d+)\s*(?:count|pcs|pieces|ct\b)', r'set\s+of\s+(\d+)']:
        m = re.search(pat, text)
        if m: return float(m.group(1))
    return 1.0

def _parse_row(content):
    text = content.lower()
    value = _extract_float(text, 'value:')
    unit_map = {'fl oz':1.0,'oz':1.0,'ounce':1.0,'lb':16.0,'pound':16.0,
                'kg':35.27,'gram':0.035,'grams':0.035,'g ':0.035,
                'ml':0.034,'liter':33.8,'litre':33.8,
                'count':1.0,'pack':1.0,'piece':1.0,'pcs':1.0}
    unit_score = next((mult for unit, mult in unit_map.items() if unit in text), 0.0)
    pack_qty = _extract_pack(text)
    title_line = next((l for l in content.split('\n') if 'item name:' in l.lower() or l.strip()), '')
    title_words, title_chars = len(title_line.split()), len(title_line)
    total_chars, total_words = len(content), len(content.split())
    digit_ratio = sum(c.isdigit() for c in content) / max(total_chars, 1)
    brands = ['apple','samsung','sony','lg','hp','dell','nike','adidas',
              'amazon','google','microsoft','cisco','bosch','philips']
    brand_hit = float(any(b in text for b in brands))
    cats = {'electronic':1,'cable':1,'adapter':1,'charger':1,'food':2,'sauce':2,
            'coffee':2,'tea':2,'clothing':3,'shirt':3,'dress':3,'shoes':3,
            'toy':4,'game':4,'supplement':5,'vitamin':5}
    category = next((v for k, v in cats.items() if k in text), 0)
    return [value, unit_score, pack_qty, title_words, title_chars,
            total_chars, total_words, digit_ratio, brand_hit, float(category),
            float(any(k in text for k in ['inch','"','cm','mm','size'])),
            float(any(k in text for k in ['oz','lb','gram','kg'])),
            float(any(k in text for k in ['ml','liter','gallon','fl oz'])),
            value * max(pack_qty, 1), np.log1p(value), np.log1p(total_words)]

print('Feature extraction functions defined')

In [ ]:
# ── Step 1: Load data ─────────────────────────────────────────────────────────
print('[1/4] Loading train.csv ...')
df = pd.read_csv('train.csv')
print(f'      {len(df):,} rows loaded')
y = np.log1p(df['price'].values.astype(np.float64))

In [ ]:
# ── Step 2: Text features (16) via existing scaler ────────────────────────────
print('[2/4] Extracting text features ...')
t0 = time.time()
rows = [_parse_row(c) for c in df['catalog_content'].fillna('')]
X_text_raw = np.array(rows, dtype=np.float32)
print(f'      Raw text features: {X_text_raw.shape}  ({time.time()-t0:.1f}s)')

with open('scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)
X_text = scaler.transform(X_text_raw)
print(f'      Scaled: {X_text.shape}')

In [ ]:
# ── Step 3: BERT embeddings (384) — fast on GPU ───────────────────────────────
print('[3/4] Encoding BERT embeddings ...')
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'      Device: {device.upper()}')

bert = SentenceTransformer('all-MiniLM-L6-v2', device=device)
texts = df['catalog_content'].fillna('').tolist()

t1 = time.time()
X_bert = bert.encode(texts, batch_size=512, show_progress_bar=True,
                     convert_to_numpy=True).astype(np.float32)
print(f'      BERT embeddings: {X_bert.shape}  ({time.time()-t1:.1f}s)')

X = np.hstack([X_text, X_bert])
print(f'      Combined: {X.shape}  (must be 400)')
assert X.shape[1] == 400, f'Expected 400 features, got {X.shape[1]}'

In [ ]:
# ── Step 4: Train GradientBoostingRegressor ───────────────────────────────────
print('[4/4] Training GradientBoostingRegressor ...')
print('      n_estimators=150, max_depth=5, lr=0.1')

model = GradientBoostingRegressor(
    n_estimators=150,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    random_state=RANDOM_SEED,
    verbose=1,
)
t2 = time.time()
model.fit(X, y)
print(f'\nDone in {time.time()-t2:.1f}s')
print(f'n_features_in_: {model.n_features_in_}  (must be 400)')

In [ ]:
# ── Save & download ───────────────────────────────────────────────────────────
with open('gradient_boosting_model.pkl', 'wb') as f:
    pickle.dump(model, f)
print('Saved gradient_boosting_model.pkl')

files.download('gradient_boosting_model.pkl')
print('\nDONE!')
print('Place gradient_boosting_model.pkl into your local models/ folder')
print('then restart app.py — all 5 models will be active.')